# 04. Feature Engineering & ML Matching Model

**Team Role**: Member 3

### Purpose:
This notebook covers the extraction of pairwise similarity features (name, address, token, character, country) from candidate pairs, model training, cross-validation, and decision threshold calibration aimed at maximizing the competition F0.5 score.

In [1]:
# ============================================
# MEMBER 3 - ITERATION 1
# Feature Engineering & ML Matching
# ============================================

import os
import sys
import pandas as pd
import numpy as np

print("Python environment ready!")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Python environment ready!
Pandas version: 2.2.3
NumPy version: 2.1.3


In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

zip_path = "/content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/6ab10eb3b23ba_student_resource.zip"

print("ZIP exists:", os.path.exists(zip_path))
print("ZIP size (GB):", os.path.getsize(zip_path) / (1024**3) if os.path.exists(zip_path) else "Not found")

ZIP exists: True
ZIP size (GB): 1.0196335818618536


In [4]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/6ab10eb3b23ba_student_resource.zip"

extract_path = "/content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/extracted"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete!")
print("Extracted to:", extract_path)

Extraction complete!
Extracted to: /content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/extracted


In [5]:
import os

for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, "").count(os.sep)

    if level > 3:
        continue

    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:10]:
        print(f"{indent}    {file}")

extracted/
    student_resource/
        README.md
        .DS_Store
        Documentation_template.md
        dataset/
            .DS_Store
            test/
                test_source3.tsv
                test_source1.tsv
                test_source2.tsv
            train/
                train_ground_truth.tsv
                train_source1.tsv
                train_source2.tsv
                train_source3.tsv
        utils/
            validate_submission.py
    __MACOSX/
        ._student_resource
        student_resource/
            ._utils
            ._Documentation_template.md
            ._.DS_Store
            ._README.md
            ._dataset
            dataset/
                ._test
                ._train
                ._.DS_Store
            utils/
                ._validate_submission.py


In [6]:
# ============================================
# Challenge dataset paths
# ============================================

BASE_PATH = "/content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/extracted/student_resource"

TRAIN_PATH = f"{BASE_PATH}/dataset/train"
TEST_PATH = f"{BASE_PATH}/dataset/test"

print("Training path:", TRAIN_PATH)
print("Test path:", TEST_PATH)

Training path: /content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/extracted/student_resource/dataset/train
Test path: /content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/extracted/student_resource/dataset/test


In [7]:
import pandas as pd

train_source1 = pd.read_csv(
    f"{TRAIN_PATH}/train_source1.tsv",
    sep="\t",
    dtype=str
)

train_source2 = pd.read_csv(
    f"{TRAIN_PATH}/train_source2.tsv",
    sep="\t",
    dtype=str
)

train_source3 = pd.read_csv(
    f"{TRAIN_PATH}/train_source3.tsv",
    sep="\t",
    dtype=str
)

train_ground_truth = pd.read_csv(
    f"{TRAIN_PATH}/train_ground_truth.tsv",
    sep="\t",
    dtype=str
)

print("Source 1:", train_source1.shape)
print("Source 2:", train_source2.shape)
print("Source 3:", train_source3.shape)
print("Ground truth:", train_ground_truth.shape)

Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)
Ground truth: (2206821, 2)


In [8]:
print("SOURCE 1 COLUMNS")
print(train_source1.columns.tolist())

print("\nSOURCE 2 COLUMNS")
print(train_source2.columns.tolist())

print("\nSOURCE 3 COLUMNS")
print(train_source3.columns.tolist())

print("\nGROUND TRUTH COLUMNS")
print(train_ground_truth.columns.tolist())

SOURCE 1 COLUMNS
['entity_id', 'business_name', 'business_address', 'country']

SOURCE 2 COLUMNS
['entity_id', 'business_name', 'business_address', 'country']

SOURCE 3 COLUMNS
['entity_id', 'business_name', 'business_address', 'country']

GROUND TRUTH COLUMNS
['source1_entity_id', 'matched_entity_ids']


In [9]:
display(train_source1.head(5))

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [10]:
display(train_source2.head(5))

,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


In [11]:
display(train_source3.head(5))

,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India


In [12]:
display(train_ground_truth.head(10))

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [13]:
# ============================================
# Member 3 - Development sample
# ============================================

DEV_N = 10_000

dev_source1 = train_source1.head(DEV_N).copy()

print("Development Source 1:", dev_source1.shape)
print("Full Source 1:", train_source1.shape)

Development Source 1: (10000, 4)
Full Source 1: (2206821, 4)


In [14]:
dev_source2 = train_source2.copy()
dev_source3 = train_source3.copy()

print("Source 2:", dev_source2.shape)
print("Source 3:", dev_source3.shape)

Source 2: (5034616, 4)
Source 3: (5285603, 4)


In [15]:
%cd /content

!git clone -b member3-model https://github.com/niharikagadhiraju12-boop/DataFlux.git

/content
Cloning into 'DataFlux'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 62 (delta 13), reused 51 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 8.93 MiB | 15.44 MiB/s, done.
Resolving deltas: 100% (13/13), done.


In [16]:
import sys

PROJECT_SRC = "/content/DataFlux/code/business_entity_resolution/src"

if PROJECT_SRC not in sys.path:
    sys.path.insert(0, PROJECT_SRC)

print("Project source path added:")
print(PROJECT_SRC)

Project source path added:
/content/DataFlux/code/business_entity_resolution/src


In [17]:
from blocking import generate_candidates

print("Member 2 blocking module imported successfully!")

Member 2 blocking module imported successfully!


In [18]:
import inspect

print(inspect.signature(generate_candidates))

(source1_df: pandas.core.frame.DataFrame, target_df: pandas.core.frame.DataFrame, use_rare_tokens: bool = True, use_tfidf: bool = True, use_address_tokens: bool = True, k_top: int = 25, min_similarity: float = 0.18, max_token_freq: int = 150, max_addr_freq: int = 30) -> pandas.core.frame.DataFrame


In [19]:
import inspect

source_code = inspect.getsource(generate_candidates)
print(source_code)

def generate_candidates(
    source1_df: pd.DataFrame,
    target_df: pd.DataFrame,
    use_rare_tokens: bool = True,
    use_tfidf: bool = True,
    use_address_tokens: bool = True,
    k_top: int = 25,
    min_similarity: float = 0.18,
    max_token_freq: int = 150,
    max_addr_freq: int = 30,
) -> pd.DataFrame:
    """
    Execute the multi-pass blocking pipeline:
        1. Country partitioning
        2. Rare name-token blocking
        3. Character n-gram TF-IDF retrieval
        4. Distinctive address-token blocking (optional)
        5. Union and deduplication
    
    Guarantees:
        - Candidates are strictly S1-S2 or S1-S3 (no S1-S1 self-matches).
        - Multiple candidates per S1 are supported.
        - Output is deterministically sorted for reproducibility.

    Parameters:
        source1_df: DataFrame containing Source 1 records.
        target_df: DataFrame containing Target (Source 2 and/or Source 3) records.
        use_rare_tokens: Whether to enable rare toke

In [20]:
TEST_S1 = dev_source1.head(1000).copy()
TEST_S2 = dev_source2.head(100_000).copy()

print("Test Source 1:", TEST_S1.shape)
print("Test Source 2:", TEST_S2.shape)

Test Source 1: (1000, 4)
Test Source 2: (100000, 4)


In [21]:
candidates_test = generate_candidates(
    source1_df=TEST_S1,
    target_df=TEST_S2
)

print("Candidate pairs generated:", len(candidates_test))
print(candidates_test.head())

Candidate pairs generated: 69039
  source1_entity_id candidate_entity_id
0      S1-100146655        S2-138958741
1      S1-100146655        S2-165008290
2      S1-100146655        S2-226521083
3      S1-100146655        S2-229962546
4      S1-100146655        S2-234426829


In [22]:
# Prepare ground truth for the 1,000 Source 1 development records

GT_TEST = train_ground_truth[
    train_ground_truth["source1_entity_id"].isin(TEST_S1["entity_id"])
].copy()

print("Ground-truth rows:", len(GT_TEST))
print(GT_TEST.head())

Ground-truth rows: 1000
     source1_entity_id                                 matched_entity_ids
1371      S1-844896591                           S2-362218588,S3-11966366
2917      S1-548116192                                                NaN
5400      S1-378978603             S2-319300693,S3-948532405,S3-921016187
7826      S1-447452795                          S2-995233298,S3-366674728
9668      S1-401761505  S2-422370961,S2-415623223,S3-17464132,S3-16723...


In [23]:
# Build the set of true Source 1 -> Source 2 pairs

true_s2_pairs = set()

for _, row in GT_TEST.iterrows():
    matched_ids = row["matched_entity_ids"]

    if pd.isna(matched_ids):
        continue

    for entity_id in str(matched_ids).split(","):
        entity_id = entity_id.strip()

        if entity_id.startswith("S2-"):
            true_s2_pairs.add(
                (row["source1_entity_id"], entity_id)
            )

# Convert generated candidates into a set for fast lookup
candidate_pairs_set = set(
    zip(
        candidates_test["source1_entity_id"],
        candidates_test["candidate_entity_id"]
    )
)

# Count how many true S2 pairs were captured
captured_s2_pairs = true_s2_pairs.intersection(candidate_pairs_set)

print("Total true S2 pairs:", len(true_s2_pairs))
print("Captured true S2 pairs:", len(captured_s2_pairs))

if len(true_s2_pairs) > 0:
    recall = len(captured_s2_pairs) / len(true_s2_pairs)
    print("S2 candidate recall:", round(recall, 4))

Total true S2 pairs: 1706
Captured true S2 pairs: 32
S2 candidate recall: 0.0188


In [24]:
# Check how many true S2 matches are actually present
# inside our 100,000-row test target dataset.

test_s2_ids = set(TEST_S2["entity_id"])

true_s2_in_test_target = {
    pair for pair in true_s2_pairs
    if pair[1] in test_s2_ids
}

print("True S2 pairs:", len(true_s2_pairs))
print("True S2 pairs present in TEST_S2:", len(true_s2_in_test_target))

if len(true_s2_pairs) > 0:
    print(
        "Fraction of true S2 pairs available to blocking:",
        round(len(true_s2_in_test_target) / len(true_s2_pairs), 4)
    )

True S2 pairs: 1706
True S2 pairs present in TEST_S2: 32
Fraction of true S2 pairs available to blocking: 0.0188


In [25]:
import psutil

memory = psutil.virtual_memory()

print("Total RAM (GB):", round(memory.total / (1024**3), 2))
print("Available RAM (GB):", round(memory.available / (1024**3), 2))
print("Used RAM (GB):", round(memory.used / (1024**3), 2))

Total RAM (GB): 12.67
Available RAM (GB): 7.18
Used RAM (GB): 5.18


In [26]:
TEST_S2_LARGE = dev_source2.head(1_000_000).copy()

print("Large test Source 2:", TEST_S2_LARGE.shape)

Large test Source 2: (1000000, 4)


In [27]:
test_s2_large_ids = set(TEST_S2_LARGE["entity_id"])

true_s2_in_large_target = {
    pair for pair in true_s2_pairs
    if pair[1] in test_s2_large_ids
}

print("Total true S2 pairs:", len(true_s2_pairs))
print("True S2 pairs present in 1M target:", len(true_s2_in_large_target))

print(
    "Fraction of true S2 pairs available:",
    round(len(true_s2_in_large_target) / len(true_s2_pairs), 4)
)

Total true S2 pairs: 1706
True S2 pairs present in 1M target: 348
Fraction of true S2 pairs available: 0.204


In [28]:
from blocking import tfidf_name_block
import inspect

print(inspect.signature(tfidf_name_block))
print()
print(inspect.getsource(tfidf_name_block))

(source1_df: pandas.core.frame.DataFrame, target_df: pandas.core.frame.DataFrame, k_top: int = 25, min_similarity: float = 0.18, chunk_size: int = 500, ngram_range: Tuple[int, int] = (3, 4)) -> Set[Tuple[str, str]]

def tfidf_name_block(
    source1_df: pd.DataFrame,
    target_df: pd.DataFrame,
    k_top: int = 25,
    min_similarity: float = 0.18,
    chunk_size: int = 500,
    ngram_range: Tuple[int, int] = (3, 4),
) -> Set[Tuple[str, str]]:
    """
    Generate candidate pairs via character n-gram TF-IDF cosine similarity.

    Why: Character n-grams are robust against misspellings ('Wilblims' vs 'Williams'),
    diacritics/accents ('Nónet' vs 'Nonet'), concatenations ('maurewilliamscolombier.com'),
    and word transpositions.

    Uses batched matrix multiplication to ensure O(chunk_size) memory usage.

    Parameters:
        source1_df: Preprocessed Source 1 records.
        target_df: Preprocessed Target records.
        k_top: Number of top candidate matches to retrieve per S

In [29]:
TEST_S1_SMALL = TEST_S1.head(100).copy()

print("Small Source 1:", TEST_S1_SMALL.shape)
print("Large Source 2:", TEST_S2_LARGE.shape)

Small Source 1: (100, 4)
Large Source 2: (1000000, 4)


In [30]:
from blocking import normalize_for_blocking

TEST_S1_SMALL_NORM = normalize_for_blocking(TEST_S1_SMALL)
TEST_S2_LARGE_NORM = normalize_for_blocking(TEST_S2_LARGE)

print("Normalized Source 1 columns:")
print(TEST_S1_SMALL_NORM.columns.tolist())

print("\nNormalized Source 2 columns:")
print(TEST_S2_LARGE_NORM.columns.tolist())

Normalized Source 1 columns:
['entity_id', 'business_name', 'business_address', 'country', 'norm_name', 'norm_address', 'norm_country']

Normalized Source 2 columns:
['entity_id', 'business_name', 'business_address', 'country', 'norm_name', 'norm_address', 'norm_country']


In [31]:
tfidf_test = tfidf_name_block(
    source1_df=TEST_S1_SMALL_NORM,
    target_df=TEST_S2_LARGE_NORM,
    k_top=25,
    min_similarity=0.18,
    chunk_size=100
)

print("TF-IDF candidate pairs:", len(tfidf_test))

TF-IDF candidate pairs: 2500


In [32]:
import features

print("Features module imported successfully!")
print(features.__file__)

Features module imported successfully!
/content/DataFlux/code/business_entity_resolution/src/features.py


In [33]:
import inspect

print(inspect.getsource(features))

"""
Feature Engineering Module for Business Entity Resolution.

Responsibilities:
- Extract pairwise comparison features for candidate entity pairs
- Produce numerical features for ML-based entity matching
"""

from typing import Any

import pandas as pd
from difflib import SequenceMatcher


def text_normalize(value: Any) -> str:
    """Convert a value to a safe normalized string."""
    if pd.isna(value):
        return ""
    return str(value).strip().lower()


def token_jaccard(a: Any, b: Any) -> float:
    """Calculate Jaccard similarity between whitespace-separated tokens."""
    a_tokens = set(text_normalize(a).split())
    b_tokens = set(text_normalize(b).split())

    if not a_tokens and not b_tokens:
        return 1.0

    if not a_tokens or not b_tokens:
        return 0.0

    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)


def edit_similarity(a: Any, b: Any) -> float:
    """Calculate normalized character-level similarity."""
    a = text_normalize(a)
    b = 

In [34]:
import importlib.util

print("rapidfuzz installed:",
      importlib.util.find_spec("rapidfuzz") is not None)

print("sklearn installed:",
      importlib.util.find_spec("sklearn") is not None)

rapidfuzz installed: False
sklearn installed: True


In [35]:
import re
import numpy as np
import pandas as pd
from difflib import SequenceMatcher


def text_normalize(value):
    """Convert a value to a safe normalized string."""
    if pd.isna(value):
        return ""
    return str(value).strip().lower()


def token_jaccard(a, b):
    """Jaccard similarity between whitespace-separated token sets."""
    a_tokens = set(text_normalize(a).split())
    b_tokens = set(text_normalize(b).split())

    if not a_tokens and not b_tokens:
        return 1.0

    if not a_tokens or not b_tokens:
        return 0.0

    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)


def edit_similarity(a, b):
    """Normalized character-level similarity."""
    a = text_normalize(a)
    b = text_normalize(b)

    if not a and not b:
        return 1.0

    if not a or not b:
        return 0.0

    return SequenceMatcher(None, a, b).ratio()


def length_difference(a, b):
    """Absolute difference in character lengths."""
    a = text_normalize(a)
    b = text_normalize(b)

    return abs(len(a) - len(b))


def extract_pair_features(
    candidate_pairs,
    source_a,
    source_b,
    **kwargs
):
    """
    Create numerical similarity features for candidate entity pairs.
    """

    # Keep only the columns needed for joining.
    a = source_a[
        ["entity_id", "business_name", "business_address", "country"]
    ].copy()

    b = source_b[
        ["entity_id", "business_name", "business_address", "country"]
    ].copy()

    # Rename columns so Source A and Source B are clearly distinguished.
    a = a.rename(columns={
        "entity_id": "source1_entity_id",
        "business_name": "name_a",
        "business_address": "address_a",
        "country": "country_a",
    })

    b = b.rename(columns={
        "entity_id": "candidate_entity_id",
        "business_name": "name_b",
        "business_address": "address_b",
        "country": "country_b",
    })

    # Attach the actual entity attributes to every candidate pair.
    df = candidate_pairs.merge(
        a,
        on="source1_entity_id",
        how="left"
    ).merge(
        b,
        on="candidate_entity_id",
        how="left"
    )

    # Name features
    df["name_exact"] = (
        df["name_a"].fillna("").astype(str).str.strip().str.lower()
        ==
        df["name_b"].fillna("").astype(str).str.strip().str.lower()
    ).astype(int)

    df["name_jaccard"] = [
        token_jaccard(a, b)
        for a, b in zip(df["name_a"], df["name_b"])
    ]

    df["name_edit_similarity"] = [
        edit_similarity(a, b)
        for a, b in zip(df["name_a"], df["name_b"])
    ]

    df["name_length_diff"] = [
        length_difference(a, b)
        for a, b in zip(df["name_a"], df["name_b"])
    ]

    # Address features
    df["address_exact"] = (
        df["address_a"].fillna("").astype(str).str.strip().str.lower()
        ==
        df["address_b"].fillna("").astype(str).str.strip().str.lower()
    ).astype(int)

    df["address_jaccard"] = [
        token_jaccard(a, b)
        for a, b in zip(df["address_a"], df["address_b"])
    ]

    df["address_edit_similarity"] = [
        edit_similarity(a, b)
        for a, b in zip(df["address_a"], df["address_b"])
    ]

    df["address_length_diff"] = [
        length_difference(a, b)
        for a, b in zip(df["address_a"], df["address_b"])
    ]

    # Country feature
    country_a = df["country_a"].fillna("").astype(str).str.strip().str.lower()
    country_b = df["country_b"].fillna("").astype(str).str.strip().str.lower()

    df["country_match"] = (
        (country_a != "") &
        (country_b != "") &
        (country_a == country_b)
    ).astype(int)

    # Token-count features
    df["name_token_count_diff"] = [
        abs(
            len(text_normalize(a).split())
            -
            len(text_normalize(b).split())
        )
        for a, b in zip(df["name_a"], df["name_b"])
    ]

    df["address_token_count_diff"] = [
        abs(
            len(text_normalize(a).split())
            -
            len(text_normalize(b).split())
        )
        for a, b in zip(df["address_a"], df["address_b"])
    ]

    return df

In [36]:
features_test = extract_pair_features(
    candidate_pairs=candidates_test,
    source_a=TEST_S1,
    source_b=TEST_S2
)

print("Feature dataframe shape:", features_test.shape)
print("\nFeature columns:")
print(features_test.columns.tolist())

print("\nFirst 5 rows:")
display(features_test.head())

Feature dataframe shape: (69039, 19)

Feature columns:
['source1_entity_id', 'candidate_entity_id', 'name_a', 'address_a', 'country_a', 'name_b', 'address_b', 'country_b', 'name_exact', 'name_jaccard', 'name_edit_similarity', 'name_length_diff', 'address_exact', 'address_jaccard', 'address_edit_similarity', 'address_length_diff', 'country_match', 'name_token_count_diff', 'address_token_count_diff']

First 5 rows:


,source1_entity_id,candidate_entity_id,name_a,address_a,country_a,name_b,address_b,country_b,name_exact,name_jaccard,name_edit_similarity,name_length_diff,address_exact,address_jaccard,address_edit_similarity,address_length_diff,country_match,name_token_count_diff,address_token_count_diff
0,S1-100146655,S2-138958741,Bricklayers Local Union 428,"4268 Davis Hill Road, Scio, NY",US,Bricklayers Union Local No,"200 CLARK LOOP, LOCKHAT, TX",US,0,0.600000,0.754717,1,0,0.0,0.385965,3,1,0,1
1,S1-100146655,S2-165008290,Bricklayers Local Union 428,"4268 Davis Hill Road, Scio, NY",US,BRICKLAYERS 481 LOCAL,"125 115ND STREET, VILLAGE OF PLEASANT PRAIRIE, WI",US,0,0.400000,0.708333,6,0,0.0,0.329114,19,1,1,2
2,S1-100146655,S2-226521083,Bricklayers Local Union 428,"4268 Davis Hill Road, Scio, NY",US,Electricians Local Union 420 Inc,"149 23RD AVENUE, DICKINSON, ND",US,0,0.285714,0.677966,5,0,0.0,0.333333,0,1,1,1
3,S1-100146655,S2-229962546,Bricklayers Local Union 428,"4268 Davis Hill Road, Scio, NY",US,Painters Union Local No,"0216 W WOODLAND AVE, UNDERWOOD, MN",US,0,0.333333,0.520000,4,0,0.0,0.406250,4,1,0,0
4,S1-100146655,S2-234426829,Bricklayers Local Union 428,"4268 Davis Hill Road, Scio, NY",US,Bricklayers Local 197 LLC,"PHOENIX, AZ, 3650 ORANGE DR",US,0,0.333333,0.730769,2,0,0.0,0.140351,3,1,0,1


In [37]:
# Create binary labels for the candidate pairs

features_test["label"] = [
    int(
        (s1_id, candidate_id) in true_s2_pairs
    )
    for s1_id, candidate_id in zip(
        features_test["source1_entity_id"],
        features_test["candidate_entity_id"]
    )
]

print("Total candidate pairs:", len(features_test))
print("Positive matches:", features_test["label"].sum())
print("Negative pairs:", (features_test["label"] == 0).sum())

print("\nLabel distribution:")
print(features_test["label"].value_counts())

Total candidate pairs: 69039
Positive matches: 32
Negative pairs: 69007

Label distribution:
label
0    69007
1       32
Name: count, dtype: int64


In [38]:
import model

print("Model module:")
print(model.__file__)

print("\nExisting model.py:")
print(inspect.getsource(model))

Model module:
/content/DataFlux/code/business_entity_resolution/src/model.py

Existing model.py:
"""
Matching Model Module for Business Entity Resolution.

Responsibilities:
- Training and configuring the classification model
- Generating match probabilities
- Generating optional threshold-based predictions
- Saving and loading trained model artifacts
"""

from pathlib import Path
from typing import Any, Dict, Optional, Union

import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression


class EntityMatchingModel:
    """
    Wrapper for a binary classification model used for
    business entity matching.
    """

    def __init__(
        self,
        model_params: Optional[Dict[str, Any]] = None,
    ) -> None:
        """
        Initialize the matching model.

        Parameters
        ----------
        model_params:
            Optional Logistic Regression hyperparameters.
        """

        default_params = {
            "class_weight": "balanced",


In [39]:
from sklearn.linear_model import LogisticRegression


# Numerical features used by the first model
FEATURE_COLUMNS = [
    "name_exact",
    "name_jaccard",
    "name_edit_similarity",
    "name_length_diff",
    "address_exact",
    "address_jaccard",
    "address_edit_similarity",
    "address_length_diff",
    "country_match",
    "name_token_count_diff",
    "address_token_count_diff",
]

X_test = features_test[FEATURE_COLUMNS].copy()
y_test = features_test["label"].copy()

print("Feature matrix shape:", X_test.shape)
print("Label shape:", y_test.shape)
print("Positive labels:", int(y_test.sum()))
print("Negative labels:", int((y_test == 0).sum()))

Feature matrix shape: (69039, 11)
Label shape: (69039,)
Positive labels: 32
Negative labels: 69007


In [40]:
# First ML model for entity matching
# This is a smoke test using the current candidate sample.

matching_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

matching_model.fit(X_test, y_test)

print("Model training completed!")

Model training completed!


In [41]:
# Generate probability that each candidate pair is a true match

match_probabilities = matching_model.predict_proba(X_test)[:, 1]

features_test["match_probability"] = match_probabilities

print("Probability generation completed!")

print("\nProbability statistics:")
print(features_test["match_probability"].describe())

print("\nTop 10 candidate pairs by probability:")
display(
    features_test[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "match_probability",
            "label"
        ]
    ]
    .sort_values("match_probability", ascending=False)
    .head(10)
)

Probability generation completed!

Probability statistics:
count    6.903900e+04
mean     1.394782e-02
std      8.215436e-02
min      6.755243e-24
25%      3.513752e-07
50%      1.524612e-05
75%      3.927716e-04
max      1.000000e+00
Name: match_probability, dtype: float64

Top 10 candidate pairs by probability:


,source1_entity_id,candidate_entity_id,match_probability,label
27297,S1-452691275,S2-320133796,1.0,1
59092,S1-877733833,S2-974050437,1.0,1
57029,S1-842881175,S2-523911840,1.0,1
567,S1-104226194,S2-894147929,1.0,1
33363,S1-533701421,S2-434618903,1.0,1
11005,S1-220020462,S2-342952603,1.0,1
55759,S1-814475089,S2-238559336,1.0,1
20787,S1-376690332,S2-787570292,1.0,1
12037,S1-236312382,S2-69010626,1.0,1
51508,S1-761585587,S2-801229590,1.0,1


In [42]:
# Prepare the probability output for Member 4

model_output = features_test[
    [
        "source1_entity_id",
        "candidate_entity_id",
        "match_probability"
    ]
].copy()

print("Model output shape:", model_output.shape)
display(model_output.head())

# Save the smoke-test probabilities
MODEL_OUTPUT_PATH = (
    "/content/drive/MyDrive/"
    "Amazon_ML_Challenge_2026/"
    "student_resource/"
    "member3_model_probabilities_smoke_test.tsv"
)

model_output.to_csv(
    MODEL_OUTPUT_PATH,
    sep="\t",
    index=False
)

print("\nSaved to:")
print(MODEL_OUTPUT_PATH)

Model output shape: (69039, 3)


,source1_entity_id,candidate_entity_id,match_probability
0,S1-100146655,S2-138958741,0.448575
1,S1-100146655,S2-165008290,0.002390
2,S1-100146655,S2-226521083,0.001084
3,S1-100146655,S2-229962546,0.000786
4,S1-100146655,S2-234426829,0.000597



Saved to:
/content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/member3_model_probabilities_smoke_test.tsv


In [43]:
import importlib
import features

importlib.reload(features)

print("Updated features.py loaded successfully!")
print(features.extract_pair_features)

Updated features.py loaded successfully!
<function extract_pair_features at 0x78380a8e9d00>


In [44]:
import importlib
import model

importlib.reload(model)

print("Updated model.py loaded successfully!")
print(model.EntityMatchingModel)

Updated model.py loaded successfully!
<class 'model.EntityMatchingModel'>


In [45]:
from model import EntityMatchingModel

# Create the project-level matching model
entity_model = EntityMatchingModel()

# Train on our smoke-test feature matrix
entity_model.fit(
    X_test,
    y_test
)

print("EntityMatchingModel training completed!")

EntityMatchingModel training completed!


In [46]:
project_probabilities = entity_model.predict_proba(X_test)[:, 1]

print("Probability generation completed!")
print("Number of probabilities:", len(project_probabilities))
print("Minimum probability:", project_probabilities.min())
print("Maximum probability:", project_probabilities.max())

Probability generation completed!
Number of probabilities: 69039
Minimum probability: 6.75524314044976e-24
Maximum probability: 1.0


In [47]:
# Test threshold-based predictions
predictions = entity_model.predict(
    X_test,
    threshold=0.5
)

print("Prediction count:", len(predictions))
print("Predicted matches:", int(predictions.sum()))
print("Predicted non-matches:", int((predictions == 0).sum()))

Prediction count: 69039
Predicted matches: 591
Predicted non-matches: 68448


In [48]:
MODEL_PATH = "/content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/entity_matching_model.joblib"

# Save the trained model
entity_model.save(MODEL_PATH)

print("Model saved to:")
print(MODEL_PATH)

Model saved to:
/content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/entity_matching_model.joblib


In [49]:
# Load the saved model
loaded_model = EntityMatchingModel.load(MODEL_PATH)

print("Model loaded successfully!")

# Generate probabilities from the loaded model
loaded_probabilities = loaded_model.predict_proba(X_test)[:, 1]

# Verify that the loaded model gives the same probabilities
same_predictions = np.allclose(
    project_probabilities,
    loaded_probabilities
)

print("Probabilities match original model:", same_predictions)

Model loaded successfully!
Probabilities match original model: True


In [50]:
%cd /content/DataFlux

!git status

/content/DataFlux
On branch member3-model
Your branch is up to date with 'origin/member3-model'.

nothing to commit, working tree clean


In [51]:
%cd /content/DataFlux

!git add code/business_entity_resolution/src/features.py
!git add code/business_entity_resolution/src/model.py

!git commit -m "Implement Member 3 features and matching model"

/content/DataFlux
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@9283bd130c56.(none)')


In [52]:
%cd /content/DataFlux

!git config user.name "Amrutha"
!git config user.email "bamrutha1876@gmail.com"

/content/DataFlux


In [53]:
!git config user.name
!git config user.email

Amrutha
bamrutha1876@gmail.com


In [54]:
%cd /content/DataFlux

!git add code/business_entity_resolution/src/features.py
!git add code/business_entity_resolution/src/model.py

!git commit -m "Implement Member 3 features and matching model"

/content/DataFlux
On branch member3-model
Your branch is up to date with 'origin/member3-model'.

nothing to commit, working tree clean


In [55]:
%cd /content/DataFlux

!git push origin member3-model

/content/DataFlux
fatal: could not read Username for 'https://github.com': No such device or address


In [56]:
import os
import subprocess
from getpass import getpass

# Go to your repository
os.chdir("/content/DataFlux")

# Enter your GitHub username
github_username = input("GitHub username: ")

# Enter the PAT securely — it will NOT be displayed
github_token = getpass("GitHub PAT (input hidden): ")

# Keep the token only in this Colab process
os.environ["GITHUB_TOKEN"] = github_token

# Configure a temporary Git credential helper.
# The token itself is NOT written into the Git config.
helper = (
    '!f() { '
    'echo username=$GITHUB_USERNAME; '
    'echo password=$GITHUB_TOKEN; '
    '}; f'
)

os.environ["GITHUB_USERNAME"] = github_username

subprocess.run(
    ["git", "config", "--local", "credential.helper", helper],
    check=True
)

# Push the branch
result = subprocess.run(
    ["git", "push", "origin", "member3-model"],
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)

# Remove the temporary credential helper and token from this process
subprocess.run(
    ["git", "config", "--local", "--unset", "credential.helper"],
    check=False
)

os.environ.pop("GITHUB_TOKEN", None)
os.environ.pop("GITHUB_USERNAME", None)

if result.returncode == 0:
    print("\n✅ Successfully pushed member3-model!")
else:
    print("\n❌ Push failed. Send me the error message above.")

GitHub username: amruthabodduluru
GitHub PAT (input hidden): ··········

Everything up-to-date


✅ Successfully pushed member3-model!


In [57]:
import os
import subprocess
from getpass import getpass

os.chdir("/content/DataFlux")

github_username = "amruthabodduluru"
github_token = getpass("GitHub PAT (input hidden): ")

os.environ["GITHUB_USERNAME"] = github_username
os.environ["GITHUB_TOKEN"] = github_token

helper = (
    '!f() { '
    'echo username=$GITHUB_USERNAME; '
    'echo password=$GITHUB_TOKEN; '
    '}; f'
)

subprocess.run(
    ["git", "config", "--local", "credential.helper", helper],
    check=True
)

# Fetch remote changes without changing our files
result = subprocess.run(
    ["git", "fetch", "origin", "member3-model"],
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)

# Show local vs remote commits
print("\n--- LOCAL BRANCH ---")
subprocess.run(["git", "log", "--oneline", "-8", "member3-model"])

print("\n--- REMOTE BRANCH ---")
subprocess.run(["git", "log", "--oneline", "-8", "origin/member3-model"])

# Clean up credentials from this process
subprocess.run(
    ["git", "config", "--local", "--unset", "credential.helper"],
    check=False
)
os.environ.pop("GITHUB_TOKEN", None)
os.environ.pop("GITHUB_USERNAME", None)

GitHub PAT (input hidden): ··········

From https://github.com/niharikagadhiraju12-boop/DataFlux
 * branch            member3-model -> FETCH_HEAD


--- LOCAL BRANCH ---

--- REMOTE BRANCH ---


'amruthabodduluru'

In [58]:
import os
import subprocess

os.chdir("/content/DataFlux")

print("=== STATUS ===")
subprocess.run(["git", "status", "--short", "--branch"])

print("\n=== LOCAL HEAD ===")
subprocess.run(["git", "log", "-1", "--oneline", "HEAD"])

print("\n=== REMOTE HEAD ===")
subprocess.run(["git", "log", "-1", "--oneline", "origin/member3-model"])

print("\n=== COMMITS ONLY ON LOCAL ===")
subprocess.run([
    "git", "log", "--oneline",
    "origin/member3-model..HEAD"
])

print("\n=== COMMITS ONLY ON REMOTE ===")
subprocess.run([
    "git", "log", "--oneline",
    "HEAD..origin/member3-model"
])

=== STATUS ===

=== LOCAL HEAD ===

=== REMOTE HEAD ===

=== COMMITS ONLY ON LOCAL ===

=== COMMITS ONLY ON REMOTE ===


CompletedProcess(args=['git', 'log', '--oneline', 'HEAD..origin/member3-model'], returncode=0)

In [59]:
%cd /content/DataFlux

!echo "LOCAL:"
!git rev-parse --short HEAD

!echo "REMOTE:"
!git rev-parse --short origin/member3-model

!echo "BRANCH:"
!git branch --show-current

!echo "STATUS:"
!git status

/content/DataFlux
LOCAL:
8537ac5
REMOTE:
8537ac5
BRANCH:
member3-model
STATUS:
On branch member3-model
Your branch is up to date with 'origin/member3-model'.

nothing to commit, working tree clean


In [60]:
%cd /content/DataFlux
!git status

/content/DataFlux
On branch member3-model
Your branch is up to date with 'origin/member3-model'.

nothing to commit, working tree clean


In [61]:
%cd /content/DataFlux
!git commit --no-edit

/content/DataFlux
On branch member3-model
Your branch is up to date with 'origin/member3-model'.

nothing to commit, working tree clean


In [62]:
import os
import subprocess
from getpass import getpass

os.chdir("/content/DataFlux")

token = getpass("GitHub PAT (input hidden): ")

os.environ["GITHUB_USERNAME"] = "amruthabodduluru"
os.environ["GITHUB_TOKEN"] = token

helper = (
    '!f() { '
    'echo username=$GITHUB_USERNAME; '
    'echo password=$GITHUB_TOKEN; '
    '}; f'
)

subprocess.run(
    ["git", "config", "--local", "credential.helper", helper],
    check=True
)

result = subprocess.run(
    ["git", "push", "origin", "member3-model"],
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)

# Remove the temporary credential helper and token
subprocess.run(
    ["git", "config", "--local", "--unset", "credential.helper"],
    check=False
)

os.environ.pop("GITHUB_TOKEN", None)
os.environ.pop("GITHUB_USERNAME", None)

GitHub PAT (input hidden): ··········

Everything up-to-date



'amruthabodduluru'

In [63]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Paths
BASE_PATH = "/content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/extracted/student_resource"
TRAIN_PATH = f"{BASE_PATH}/dataset/train"

# Load only what we need for the split
train_source1 = pd.read_csv(
    f"{TRAIN_PATH}/train_source1.tsv",
    sep="\t",
    dtype=str
)

train_ground_truth = pd.read_csv(
    f"{TRAIN_PATH}/train_ground_truth.tsv",
    sep="\t",
    dtype=str
)

# Split by Source-1 entity (NOT by candidate pair)
source1_ids = train_source1["entity_id"].dropna().unique()

train_ids, val_ids = train_test_split(
    source1_ids,
    test_size=0.20,
    random_state=42
)

train_ids = set(train_ids)
val_ids = set(val_ids)

print("Total Source-1 entities:", len(source1_ids))
print("Training Source-1 entities:", len(train_ids))
print("Validation Source-1 entities:", len(val_ids))

# Build the actual Source-1 holdout dataframes
s1_train = train_source1[
    train_source1["entity_id"].isin(train_ids)
].copy()

s1_val = train_source1[
    train_source1["entity_id"].isin(val_ids)
].copy()

gt_train = train_ground_truth[
    train_ground_truth["source1_entity_id"].isin(train_ids)
].copy()

gt_val = train_ground_truth[
    train_ground_truth["source1_entity_id"].isin(val_ids)
].copy()

print("\nS1 train:", s1_train.shape)
print("S1 validation:", s1_val.shape)
print("GT train:", gt_train.shape)
print("GT validation:", gt_val.shape)

print("\n✅ Proper Source-1-level holdout created.")

Total Source-1 entities: 2206821
Training Source-1 entities: 1765456
Validation Source-1 entities: 441365

S1 train: (1765456, 4)
S1 validation: (441365, 4)
GT train: (1765456, 2)
GT validation: (441365, 2)

✅ Proper Source-1-level holdout created.


In [64]:
# Small integration test — intentionally limited target samples

TEST_S1 = s1_val.head(100).copy()
TEST_S2 = train_source2.head(100_000).copy()
TEST_S3 = train_source3.head(100_000).copy()

print("S1 test:", TEST_S1.shape)
print("S2 test:", TEST_S2.shape)
print("S3 test:", TEST_S3.shape)

print("\nGenerating S2 candidates...")
test_candidates_s2 = generate_candidates(
    TEST_S1,
    TEST_S2
)

print("S2 candidates:", test_candidates_s2.shape)

print("\nGenerating S3 candidates...")
test_candidates_s3 = generate_candidates(
    TEST_S1,
    TEST_S3
)

print("S3 candidates:", test_candidates_s3.shape)

test_candidates = pd.concat(
    [test_candidates_s2, test_candidates_s3],
    ignore_index=True
).drop_duplicates()

print("\nCombined candidates:", test_candidates.shape)
print("Unique S1 entities:", test_candidates["source1_entity_id"].nunique())

print("\n✅ S2 + S3 integration test completed.")

S1 test: (100, 4)
S2 test: (100000, 4)
S3 test: (100000, 4)

Generating S2 candidates...
S2 candidates: (6538, 2)

Generating S3 candidates...
S3 candidates: (6775, 2)

Combined candidates: (13313, 2)
Unique S1 entities: 100

✅ S2 + S3 integration test completed.


In [65]:
# Load the feature module from the integrated project
import sys
import pandas as pd

PROJECT_SRC = "/content/DataFlux/code/business_entity_resolution/src"

if PROJECT_SRC not in sys.path:
    sys.path.insert(0, PROJECT_SRC)

from features import extract_pair_features

# Combine S2 + S3 target records
test_targets = pd.concat(
    [TEST_S2, TEST_S3],
    ignore_index=True
).drop_duplicates(subset=["entity_id"])

# Extract Member 3 pairwise features
test_features = extract_pair_features(
    test_candidates,
    TEST_S1,
    test_targets
)

print("Feature dataframe shape:", test_features.shape)

print("\nFeature columns:")
print(test_features.columns.tolist())

print("\nFirst 5 rows:")
display(test_features.head())

print("\n✅ Member 3 feature pipeline works with S2 + S3 candidates.")

Feature dataframe shape: (13313, 19)

Feature columns:
['source1_entity_id', 'candidate_entity_id', 'name_a', 'address_a', 'country_a', 'name_b', 'address_b', 'country_b', 'name_exact', 'name_jaccard', 'name_edit_similarity', 'name_length_diff', 'address_exact', 'address_jaccard', 'address_edit_similarity', 'address_length_diff', 'country_match', 'name_token_count_diff', 'address_token_count_diff']

First 5 rows:


,source1_entity_id,candidate_entity_id,name_a,address_a,country_a,name_b,address_b,country_b,name_exact,name_jaccard,name_edit_similarity,name_length_diff,address_exact,address_jaccard,address_edit_similarity,address_length_diff,country_match,name_token_count_diff,address_token_count_diff
0,S1-106221717,S2-106256769,Aditya Sun Consultants Private Limited,"Sy.No.1009, Flat No.60206, Iind Floor, Maple B...",India,SUN DEVELOPERS PRIVATE LIMITED,"H.NO 9 , GALI NO. -3, GOKARAN WARD NO.6, MARGH...",India,0,0.500000,0.647059,8,0,0.000000,0.301887,40,1,1,0
1,S1-106221717,S2-110617122,Aditya Sun Consultants Private Limited,"Sy.No.1009, Flat No.60206, Iind Floor, Maple B...",India,Star Brothers Exports Private Limited,"NO 1009 , ROAD NO-87, CHORASI, Gujarat",India,0,0.250000,0.560000,1,0,0.000000,0.268293,88,1,0,9
2,S1-106221717,S2-117672295,Aditya Sun Consultants Private Limited,"Sy.No.1009, Flat No.60206, Iind Floor, Maple B...",India,Aditya Párts Private Limited,"DOOR NO 20. ISTM FLOOR, SBI OFFICERCOLONY, IST...",India,0,0.500000,0.757576,10,0,0.037037,0.391960,53,1,1,4
3,S1-106221717,S2-132866128,Aditya Sun Consultants Private Limited,"Sy.No.1009, Flat No.60206, Iind Floor, Maple B...",India,BABA SUN VENTURES PVT LTD,"1 , CTS 1 NO, 5 ARJUN BUILDING, PUNE, Maharashtra",India,0,0.111111,0.562500,12,0,0.000000,0.251429,77,1,0,6
4,S1-106221717,S2-133380338,Aditya Sun Consultants Private Limited,"Sy.No.1009, Flat No.60206, Iind Floor, Maple B...",India,Apex Sun Products Textiles Private Limited,"தமிழ்நாடு, NO.3-287 , 2ND FLOOR, PURASAWALKAM ...",India,0,0.375000,0.650000,4,0,0.041667,0.352941,48,1,1,6



✅ Member 3 feature pipeline works with S2 + S3 candidates.


In [66]:
# Create pair-level ground-truth labels for the 13,313 candidate pairs

# Build the set of true (Source1, target) matches
true_pairs = set()

for _, row in gt_val[gt_val["source1_entity_id"].isin(TEST_S1["entity_id"])].iterrows():
    s1_id = row["source1_entity_id"]
    matched_ids = row["matched_entity_ids"]

    if pd.isna(matched_ids) or str(matched_ids).strip() == "":
        continue

    for target_id in str(matched_ids).split(","):
        target_id = target_id.strip()
        if target_id:
            true_pairs.add((s1_id, target_id))

# Label each candidate pair
test_features["label"] = [
    int((s1, candidate) in true_pairs)
    for s1, candidate in zip(
        test_features["source1_entity_id"],
        test_features["candidate_entity_id"]
    )
]

print("Total candidate pairs:", len(test_features))
print("Positive pairs:", int(test_features["label"].sum()))
print("Negative pairs:", int((test_features["label"] == 0).sum()))

print("\nPositive rate:",
      test_features["label"].mean())

print("\n✅ Pair-level S2 + S3 ground-truth labels created.")

Total candidate pairs: 13313
Positive pairs: 3
Negative pairs: 13310

Positive rate: 0.00022534364906482386

✅ Pair-level S2 + S3 ground-truth labels created.


In [67]:
from model import EntityMatchingModel

FEATURE_COLUMNS = [
    "name_exact",
    "name_jaccard",
    "name_edit_similarity",
    "name_length_diff",
    "address_exact",
    "address_jaccard",
    "address_edit_similarity",
    "address_length_diff",
    "country_match",
    "name_token_count_diff",
    "address_token_count_diff",
]

X = test_features[FEATURE_COLUMNS].copy()
y = test_features["label"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive labels:", int(y.sum()))

# Train Member 3 model
matching_model = EntityMatchingModel()
matching_model.fit(X, y)

# Generate pair-level probabilities
test_features["match_probability"] = (
    matching_model.predict_proba(X)[:, 1]
)

print("\nProbability summary:")
print(test_features["match_probability"].describe())

print("\nTop 10 candidate pairs:")
display(
    test_features[
        [
            "source1_entity_id",
            "candidate_entity_id",
            "label",
            "match_probability"
        ]
    ]
    .sort_values("match_probability", ascending=False)
    .head(10)
)

print("\n✅ Member 3 model successfully generated S2 + S3 match probabilities.")

X shape: (13313, 11)
y shape: (13313,)
Positive labels: 3

Probability summary:
count    1.331300e+04
mean     7.421968e-03
std      5.925441e-02
min      1.287898e-89
25%      5.810947e-16
50%      1.739454e-08
75%      2.578903e-05
max      9.998350e-01
Name: match_probability, dtype: float64

Top 10 candidate pairs:


,source1_entity_id,candidate_entity_id,label,match_probability
8675,S1-369711048,S3-233375643,1,0.999835
5862,S1-883829621,S2-263304573,0,0.999579
13275,S1-990319941,S3-525707099,0,0.999272
1376,S1-236312382,S2-69010626,1,0.997387
4459,S1-740254254,S2-195865543,0,0.996955
6517,S1-990319941,S2-74209293,0,0.990074
2110,S1-356319224,S2-110974330,0,0.984638
2078,S1-356020591,S2-594625212,0,0.983086
5701,S1-866369736,S2-580478902,1,0.981149
8527,S1-349873839,S3-718476277,0,0.974376



✅ Member 3 model successfully generated S2 + S3 match probabilities.


In [68]:
from sklearn.metrics import precision_score, recall_score, fbeta_score
import numpy as np
import pandas as pd

thresholds = np.arange(0.05, 1.00, 0.05)

results = []

y_true = test_features["label"].values
probs = test_features["match_probability"].values

for threshold in thresholds:
    y_pred = (probs >= threshold).astype(int)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f05 = fbeta_score(
        y_true,
        y_pred,
        beta=0.5,
        zero_division=0
    )

    results.append({
        "threshold": threshold,
        "predicted_matches": int(y_pred.sum()),
        "precision": precision,
        "recall": recall,
        "F0.5": f05
    })

threshold_results = pd.DataFrame(results)

display(
    threshold_results.sort_values(
        "F0.5",
        ascending=False
    )
)

,threshold,predicted_matches,precision,recall,F0.5
18,0.95,14,0.214286,1.0,0.254237
17,0.90,21,0.142857,1.0,0.172414
16,0.85,24,0.125000,1.0,0.151515
15,0.80,26,0.115385,1.0,0.140187
14,0.75,31,0.096774,1.0,0.118110
13,0.70,39,0.076923,1.0,0.094340
12,0.65,42,0.071429,1.0,0.087719
11,0.60,47,0.063830,1.0,0.078534
10,0.55,51,0.058824,1.0,0.072464
9,0.50,59,0.050847,1.0,0.062762


In [69]:
import time

start = time.time()

_ = extract_pair_features(
    test_candidates,
    TEST_S1,
    test_targets
)

elapsed = time.time() - start

print(f"Current feature extraction time: {elapsed:.2f} seconds")
print(f"Candidate pairs: {len(test_candidates):,}")
print(f"Pairs/second: {len(test_candidates) / max(elapsed, 1e-9):,.0f}")

Current feature extraction time: 3.52 seconds
Candidate pairs: 13,313
Pairs/second: 3,785


In [70]:
VAL_N = 5000

S1_5K = s1_val.head(VAL_N).copy()
S2_100K = train_source2.head(100_000).copy()
S3_100K = train_source3.head(100_000).copy()

print("S1:", S1_5K.shape)
print("S2:", S2_100K.shape)
print("S3:", S3_100K.shape)

print("\nGenerating S2 candidates...")
cand_s2_5k = generate_candidates(S1_5K, S2_100K)
print("S2 candidates:", cand_s2_5k.shape)

print("\nGenerating S3 candidates...")
cand_s3_5k = generate_candidates(S1_5K, S3_100K)
print("S3 candidates:", cand_s3_5k.shape)

cand_5k = (
    pd.concat([cand_s2_5k, cand_s3_5k], ignore_index=True)
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nCombined candidates:", cand_5k.shape)

S1: (5000, 4)
S2: (100000, 4)
S3: (100000, 4)

Generating S2 candidates...
S2 candidates: (347921, 2)

Generating S3 candidates...
S3 candidates: (363435, 2)

Combined candidates: (711356, 2)


In [72]:
# Build ground-truth pair set for the 5K validation entities

gt_5k = gt_val[
    gt_val["source1_entity_id"].isin(S1_5K["entity_id"])
].copy()

true_pairs_5k = set()

for _, row in gt_5k.iterrows():
    s1_id = row["source1_entity_id"]
    matched_ids = row["matched_entity_ids"]

    if pd.isna(matched_ids) or str(matched_ids).strip() == "":
        continue

    for target_id in str(matched_ids).split(","):
        target_id = target_id.strip()
        if target_id:
            true_pairs_5k.add((s1_id, target_id))

candidate_pair_set_5k = set(
    zip(
        cand_5k["source1_entity_id"],
        cand_5k["candidate_entity_id"]
    )
)

found_true_pairs = true_pairs_5k.intersection(candidate_pair_set_5k)

print("Ground-truth pairs:", len(true_pairs_5k))
print("Candidate pairs:", len(cand_5k))
print("True pairs found in candidates:", len(found_true_pairs))

if len(true_pairs_5k) > 0:
    print(
        "Candidate recall:",
        len(found_true_pairs) / len(true_pairs_5k)
    )
else:
    print("Candidate recall: N/A")

Ground-truth pairs: 17498
Candidate pairs: 711356
True pairs found in candidates: 306
Candidate recall: 0.01748771288147217


In [73]:
ITER1_TRAIN_N = 1000
ITER1_VAL_N = 1000

iter1_train_s1 = s1_train.head(ITER1_TRAIN_N).copy()
iter1_val_s1 = s1_val.head(ITER1_VAL_N).copy()

assert set(iter1_train_s1["entity_id"]).isdisjoint(
    set(iter1_val_s1["entity_id"])
)

print("Train S1:", len(iter1_train_s1))
print("Validation S1:", len(iter1_val_s1))
print("Overlap:", len(
    set(iter1_train_s1["entity_id"]) &
    set(iter1_val_s1["entity_id"])
))

Train S1: 1000
Validation S1: 1000
Overlap: 0


In [74]:
iter1_s1 = pd.concat(
    [iter1_train_s1, iter1_val_s1],
    ignore_index=True
)

print("Total Source1 for blocking:", len(iter1_s1))

Total Source1 for blocking: 2000


In [75]:
import sys

sys.path.insert(
    0,
    "/content/DataFlux/code/business_entity_resolution"
)

from src.blocking import generate_candidates

print("generate_candidates loaded successfully!")

generate_candidates loaded successfully!


In [76]:
import pandas as pd
import numpy as np
import sys
import os

BASE_PATH = "/content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/extracted/student_resource"
TRAIN_PATH = f"{BASE_PATH}/dataset/train"
TEST_PATH = f"{BASE_PATH}/dataset/test"

train_source1 = pd.read_csv(
    f"{TRAIN_PATH}/train_source1.tsv",
    sep="\t",
    dtype=str
)

train_source2 = pd.read_csv(
    f"{TRAIN_PATH}/train_source2.tsv",
    sep="\t",
    dtype=str
)

train_source3 = pd.read_csv(
    f"{TRAIN_PATH}/train_source3.tsv",
    sep="\t",
    dtype=str
)

train_ground_truth = pd.read_csv(
    f"{TRAIN_PATH}/train_ground_truth.tsv",
    sep="\t",
    dtype=str
)

print("Source1:", train_source1.shape)
print("Source2:", train_source2.shape)
print("Source3:", train_source3.shape)
print("Ground truth:", train_ground_truth.shape)

Source1: (2206821, 4)
Source2: (5034616, 4)
Source3: (5285603, 4)
Ground truth: (2206821, 2)


In [77]:
ITER1_TRAIN_N = 1000
ITER1_VAL_N = 1000

iter1_train_s1 = train_source1.head(ITER1_TRAIN_N).copy()
iter1_val_s1 = train_source1.iloc[ITER1_TRAIN_N:ITER1_TRAIN_N + ITER1_VAL_N].copy()

assert set(iter1_train_s1["entity_id"]).isdisjoint(
    set(iter1_val_s1["entity_id"])
)

iter1_s1 = pd.concat(
    [iter1_train_s1, iter1_val_s1],
    ignore_index=True
)

print("Train S1:", len(iter1_train_s1))
print("Validation S1:", len(iter1_val_s1))
print("Total S1:", len(iter1_s1))
print("Overlap:", len(
    set(iter1_train_s1["entity_id"]) &
    set(iter1_val_s1["entity_id"])
))

Train S1: 1000
Validation S1: 1000
Total S1: 2000
Overlap: 0


In [78]:
import sys

sys.path.insert(
    0,
    "/content/DataFlux/code/business_entity_resolution"
)

from src.blocking import generate_candidates

print("generate_candidates loaded ✅")

generate_candidates loaded ✅


In [79]:
import re
import pandas as pd

def fast_normalize(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return " ".join(x.split())

# Prepare Source 1
fast_s1 = iter1_s1.copy()
fast_s1["block_name"] = fast_s1["business_name"].map(fast_normalize)
fast_s1["block_country"] = fast_s1["country"].map(fast_normalize)

# Prepare S2 + S3
fast_s2 = train_source2.copy()
fast_s3 = train_source3.copy()

for df in [fast_s2, fast_s3]:
    df["block_name"] = df["business_name"].map(fast_normalize)
    df["block_country"] = df["country"].map(fast_normalize)

print("Fast blocking keys created.")

Fast blocking keys created.


In [80]:
def make_fast_candidates(source1_df, target_df):
    # Exact normalized name + country blocking
    target_lookup = (
        target_df[
            ["entity_id", "block_name", "block_country"]
        ]
        .drop_duplicates()
    )

    pairs = source1_df[
        ["entity_id", "block_name", "block_country"]
    ].merge(
        target_lookup,
        on=["block_name", "block_country"],
        how="inner",
        suffixes=("_s1", "_target")
    )

    return (
        pairs.rename(columns={
            "entity_id_s1": "source1_entity_id",
            "entity_id_target": "candidate_entity_id"
        })[
            ["source1_entity_id", "candidate_entity_id"]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )


print("Generating fast S2 candidates...")
iter1_candidates_s2 = make_fast_candidates(
    fast_s1,
    fast_s2
)
print("S2 candidates:", len(iter1_candidates_s2))

print("\nGenerating fast S3 candidates...")
iter1_candidates_s3 = make_fast_candidates(
    fast_s1,
    fast_s3
)
print("S3 candidates:", len(iter1_candidates_s3))

iter1_candidates = pd.concat(
    [iter1_candidates_s2, iter1_candidates_s3],
    ignore_index=True
).drop_duplicates().reset_index(drop=True)

print("\nCombined candidates:", len(iter1_candidates))

Generating fast S2 candidates...
S2 candidates: 9424

Generating fast S3 candidates...
S3 candidates: 10224

Combined candidates: 19648


In [81]:
train_ids = set(iter1_train_s1["entity_id"])
val_ids = set(iter1_val_s1["entity_id"])

iter1_candidates_train = iter1_candidates[
    iter1_candidates["source1_entity_id"].isin(train_ids)
].copy()

iter1_candidates_val = iter1_candidates[
    iter1_candidates["source1_entity_id"].isin(val_ids)
].copy()

assert set(
    iter1_candidates_train["source1_entity_id"]
).isdisjoint(
    set(iter1_candidates_val["source1_entity_id"])
)

print("Train candidates:", len(iter1_candidates_train))
print("Validation candidates:", len(iter1_candidates_val))
print(
    "Source1 overlap:",
    len(
        set(iter1_candidates_train["source1_entity_id"]) &
        set(iter1_candidates_val["source1_entity_id"])
    )
)

Train candidates: 10031
Validation candidates: 9617
Source1 overlap: 0


In [82]:
def build_true_pairs(ground_truth_df):
    true_pairs = set()

    for _, row in ground_truth_df.iterrows():
        source1_id = str(row["source1_entity_id"])
        matched = row["matched_entity_ids"]

        if pd.isna(matched) or str(matched).strip() == "":
            continue

        for candidate_id in str(matched).split(","):
            candidate_id = candidate_id.strip()

            if candidate_id:
                true_pairs.add(
                    (source1_id, candidate_id)
                )

    return true_pairs


# Ground truth corresponding to our two Source1 splits
gt_train_iter1 = train_ground_truth[
    train_ground_truth["source1_entity_id"].isin(train_ids)
].copy()

gt_val_iter1 = train_ground_truth[
    train_ground_truth["source1_entity_id"].isin(val_ids)
].copy()

true_pairs_train = build_true_pairs(gt_train_iter1)
true_pairs_val = build_true_pairs(gt_val_iter1)

# Label candidate pairs
iter1_candidates_train["label"] = [
    int((str(s1), str(candidate)) in true_pairs_train)
    for s1, candidate in zip(
        iter1_candidates_train["source1_entity_id"],
        iter1_candidates_train["candidate_entity_id"]
    )
]

iter1_candidates_val["label"] = [
    int((str(s1), str(candidate)) in true_pairs_val)
    for s1, candidate in zip(
        iter1_candidates_val["source1_entity_id"],
        iter1_candidates_val["candidate_entity_id"]
    )
]

print("Train true pairs:", len(true_pairs_train))
print("Validation true pairs:", len(true_pairs_val))

print("Train positive candidates:", iter1_candidates_train["label"].sum())
print("Validation positive candidates:", iter1_candidates_val["label"].sum())

Train true pairs: 3517
Validation true pairs: 3469
Train positive candidates: 758
Validation positive candidates: 787


In [83]:
from src.features import extract_pair_features

print("Feature extractor loaded ✅")

Feature extractor loaded ✅


In [84]:
print("Extracting TRAIN features...")

iter1_train_features = extract_pair_features(
    iter1_candidates_train,
    iter1_train_s1,
    pd.concat([train_source2, train_source3], ignore_index=True)
)

print("Train feature shape:", iter1_train_features.shape)

print("\nExtracting VALIDATION features...")

iter1_val_features = extract_pair_features(
    iter1_candidates_val,
    iter1_val_s1,
    pd.concat([train_source2, train_source3], ignore_index=True)
)

print("Validation feature shape:", iter1_val_features.shape)

Extracting TRAIN features...
Train feature shape: (10031, 20)

Extracting VALIDATION features...
Validation feature shape: (9617, 20)


In [85]:
FEATURE_COLUMNS = [
    "name_exact",
    "name_jaccard",
    "name_edit_similarity",
    "name_length_diff",
    "address_exact",
    "address_jaccard",
    "address_edit_similarity",
    "address_length_diff",
    "country_match",
    "name_token_count_diff",
    "address_token_count_diff",
]

print("FEATURE_COLUMNS loaded:", len(FEATURE_COLUMNS))

FEATURE_COLUMNS loaded: 11


In [86]:
from src.model import EntityMatchingModel

X_train = iter1_train_features[FEATURE_COLUMNS]
y_train = iter1_train_features["label"]

X_val = iter1_val_features[FEATURE_COLUMNS]
y_val = iter1_val_features["label"]

print("X_train:", X_train.shape)
print("y_train positives:", int(y_train.sum()))
print("X_val:", X_val.shape)
print("y_val positives:", int(y_val.sum()))

matching_model = EntityMatchingModel()
matching_model.fit(X_train, y_train)

print("Model trained successfully ✅")

X_train: (10031, 11)
y_train positives: 758
X_val: (9617, 11)
y_val positives: 787
Model trained successfully ✅


In [87]:
iter1_val_features["match_probability"] = (
    matching_model.predict_proba(X_val)[:, 1]
)

print("Validation probabilities generated ✅")
print(
    iter1_val_features["match_probability"].describe()
)

Validation probabilities generated ✅
count    9617.000000
mean        0.144878
std         0.265325
min         0.009050
25%         0.024683
50%         0.035004
75%         0.085363
max         1.000000
Name: match_probability, dtype: float64


In [88]:
HANDOFF_PATH = (
    "/content/drive/MyDrive/"
    "Amazon_ML_Challenge_2026/student_resource/"
    "member3_iteration1_validation_probabilities.tsv"
)

member3_handoff = iter1_val_features[
    [
        "source1_entity_id",
        "candidate_entity_id",
        "label",
        "match_probability"
    ]
].copy()

member3_handoff.to_csv(
    HANDOFF_PATH,
    sep="\t",
    index=False
)

print("Member 3 handoff saved successfully ✅")
print("Path:", HANDOFF_PATH)
print("Rows:", len(member3_handoff))
print("Columns:", list(member3_handoff.columns))
print("\nPreview:")
display(member3_handoff.head())

Member 3 handoff saved successfully ✅
Path: /content/drive/MyDrive/Amazon_ML_Challenge_2026/student_resource/member3_iteration1_validation_probabilities.tsv
Rows: 9617
Columns: ['source1_entity_id', 'candidate_entity_id', 'label', 'match_probability']

Preview:


,source1_entity_id,candidate_entity_id,label,match_probability
0,S1-223283080,S2-5282695,0,0.029995
1,S1-223283080,S2-92887439,1,0.999995
2,S1-684205378,S2-949521953,0,0.028183
3,S1-684205378,S2-688975890,0,0.046802
4,S1-684205378,S2-283340383,0,0.063176


In [89]:
!ls -lh /content/DataFlux/notebooks/

total 324K
-rw-r--r-- 1 root root  94K Sep 26 03:50 01_data_analysis.ipynb
-rw-r--r-- 1 root root  45K Sep 26 03:50 02_preprocessing.ipynb
-rw-r--r-- 1 root root  21K Sep 26 03:50 03_blocking.ipynb
-rw-r--r-- 1 root root 149K Sep 26 03:50 04_model.ipynb
-rw-r--r-- 1 root root  622 Sep 26 03:50 05_evaluation.ipynb


In [90]:
%cd /content/DataFlux
!git status

/content/DataFlux
On branch member3-model
Your branch is up to date with 'origin/member3-model'.

nothing to commit, working tree clean


In [91]:
%cd /content/DataFlux
!git log -1 --format="%h %ad %s" --date=local -- notebooks/04_model.ipynb
!ls -lh notebooks/04_model.ipynb%cd /content/DataFlux
!git log -1 --format="%h %ad %s" --date=local -- notebooks/04_model.ipynb
!ls -lh notebooks/04_model.ipynb

/content/DataFlux
2c4a782 Fri Sep 25 16:48:22 2026 Implement Member 3 feature engineering and matching model
ls: cannot access 'notebooks/04_model.ipynb%cd': No such file or directory
/content/DataFlux:
total 20K
drwxr-xr-x 3 root root 4.0K Sep 26 03:50 code
-rw-r--r-- 1 root root 3.1K Sep 26 03:50 Documentation_template.md
drwxr-xr-x 2 root root 4.0K Sep 26 03:50 notebooks
drwxr-xr-x 2 root root 4.0K Sep 26 03:50 output
-rw-r--r-- 1 root root 3.9K Sep 26 03:50 README.md
2c4a782 Fri Sep 25 16:48:22 2026 Implement Member 3 feature engineering and matching model
-rw-r--r-- 1 root root 149K Sep 26 03:50 notebooks/04_model.ipynb
